In [1]:
# ============================================================================
# INPUT CHECK - runs first, before the stage below it.
#
# Every stage resolves its inputs with glob("/kaggle/input/**/<name>") and takes
# the FIRST hit alphabetically. A file attached twice is therefore not an error,
# it is a coin toss - and the order is worse than random: a folder named
# stage1-data-pipeline-OLD sorts BEFORE stage1-data-pipeline, so the copy you
# meant to retire is the one that wins, silently.
#
# This prints what will actually be read, and stops the run if anything is
# attached twice. Everything lives inside one function so it cannot collide
# with a name the stage below uses.
# ============================================================================
def _check_kaggle_inputs():
    import glob, os
    from datetime import datetime

    WANTED = {
        "unified.parquet":                     "Stage 1 corpus        (Stages 2, 4)",
        "splits.json":                         "Stage 1 splits        (Stage 2)",
        "predictions_finetuned.parquet":       "Stage 2 encoder store (Stage 2 resume, 4, 5)",
        "predictions_llm.parquet":             "Stage 3 LLM binary    (Stages 4, 5)",
        "predictions_subtype.parquet":         "Stage 3b raw subtype  (Stage 3b-repair)",
        "predictions_subtype_repaired.parquet": "Stage 3b repaired    (Stages 4, 5)",
        "tab9_evaluability.csv":               "Stage 4 gate          (Stage 5)",
    }
    # Files THIS stage actually reads. A bad version of one of these is a
    # hard stop; anything else in WANTED is checked for information only.
    CONSUMED = {"predictions_subtype.parquet", "predictions_llm.parquet"}

    print("=" * 78)
    print("ATTACHED INPUTS")
    print("=" * 78)
    problems = []
    for fname, used_by in WANTED.items():
        hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
        print(f"\n{fname}   <- {used_by}")
        if not hits:
            print("   (not attached)")
            continue
        for i, h in enumerate(hits):
            when = datetime.fromtimestamp(os.path.getmtime(h)).strftime("%Y-%m-%d %H:%M")
            mark = "  >> THIS ONE WILL BE USED" if i == 0 else "     ignored"
            print(f"   {h}\n      {os.path.getsize(h):>12,} bytes   {when}{mark}")
        if len(hits) > 1:
            problems.append(f"DUPLICATE: {fname} is attached {len(hits)} times")

    print("\n" + "=" * 78)
    print("ROW COUNTS OF WHAT WILL ACTUALLY BE READ")
    print("=" * 78)
    # A file of the right NAME can still be the wrong VERSION. The row count is
    # what tells them apart. The encoder store has two legitimate sizes
    # depending on where you are in the chain, so it is checked against both.
    EXPECTED = {
        "unified.parquet":                     ({1412}, "1,412"),
        "predictions_finetuned.parquet":       ({16900, 24280},
                                                "16,900 before Stage 2 / 24,280 after it"),
        "predictions_llm.parquet":             ({35049}, "35,049"),
        "predictions_subtype.parquet":         ({16786}, "16,786"),
        "predictions_subtype_repaired.parquet": ({16786}, "16,786"),
    }
    try:
        import pandas as pd
        for fname, (ok_values, note) in EXPECTED.items():
            hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
            if not hits:
                continue
            n = len(pd.read_parquet(hits[0]))
            # The expected values are the counts of the COMMITTED REFERENCE
            # run. A LARGER prediction store is legitimate after an API top-up
            # (the harnesses resume and add previously rate-limited items); a
            # SMALLER one is an old or partial store and must not be used.
            # unified.parquet must match exactly: requirement ids are
            # positional, so a corpus of any other size silently re-points
            # every stored prediction.
            if n in ok_values:
                flag = "OK"
            elif fname == "unified.parquet":
                flag = "<-- CORPUS SIZE CHANGED: ids are positional, do NOT run"
            elif n > max(ok_values):
                flag = ("larger than the committed reference (expected after an "
                        "API top-up; verify provenance, then update EXPECTED here)")
            else:
                flag = ("<-- SMALLER than the committed reference "
                        "(old/partial store), do not run")
            print(f"  {fname:38s} {n:>7,} rows   expected {note}   {flag}")
            # A '<--' flag on a file THIS stage actually consumes is a hard
            # stop: the old check printed 'do not run' and then ran anyway,
            # which is how hours of compute get spent against a wrong input.
            if flag.startswith("<--") and fname in CONSUMED:
                problems.append(f"BAD INPUT: {fname} - {flag.lstrip('<- ')}")
            if fname == "predictions_finetuned.parquet":
                which = ("the OLD store (correct INPUT for Stage 2 itself)" if n == 16900
                         else "the NEW store (required by Stages 4 and 5)" if n == 24280
                         else "neither size this chain produces")
                print(f"      -> this is {which}")
    except Exception as e:
        print("  (could not read:", e, ")")

    sp = sorted(glob.glob("/kaggle/input/**/splits.json", recursive=True))
    if sp:
        import json
        fams = json.load(open(sp[0]))
        n_folds = sum(len(v) for v in fams.values())
        verdict = "OK" if len(fams) == 14 else "<-- OLD Stage 1 output, re-run Stage 1 first"
        print(f"\n  splits.json: {len(fams)} families, {n_folds} folds   {verdict}")
        if verdict.startswith("<--") and "splits.json" in CONSUMED:
            problems.append("BAD INPUT: splits.json - OLD Stage 1 output "
                            f"({len(fams)} families, expected 14)")

    print("\n" + "=" * 78)
    if problems:
        for p in problems:
            print(f"  {p}")
        raise SystemExit("Fix the problems above (detach duplicates / attach "
                         "the right versions), then run again. "
                         "The stage below did NOT run.")
    print("No duplicates. Running the stage now.")
    print("=" * 78 + "\n")


_check_kaggle_inputs()


# =============================================================================
# STAGE 3b - REPAIR & RE-ANALYSIS CELL
#
# Runs AFTER the committed Stage 3b harness. Reads the existing prediction
# store and recomputes every downstream artefact correctly. It runs NO model,
# makes NO API call, and needs NO GPU. Runtime ~60 s on CPU.
#
# WHAT IT FIXES (each defect verified against the committed store):
#
#   R1  SCOPE IS RECONSTRUCTED AND STORED.
#       The harness ran each local model twice at k=0: once on the fixed CORE
#       subset and once on the FULL frame, but `scope` was never a column in
#       STORE_COLS. Downstream code therefore could not tell a 163-item API
#       evaluation from a 491-item local one and compared them directly.
#       The core id-set is recoverable exactly: it is the id-set of the k=1
#       few-shot rows, which is identical across all six local models and a
#       strict superset of every API model's ids. Verified, not assumed - the
#       cell asserts both properties and aborts if either fails.
#
#   R2  FEW-SHOT IS ACTUALLY PAIRED.
#       The harness header claimed k=0/k=1/k=2 were "paired on identical
#       items". They were not: k=0 covered the FULL frame (341/415/491) while
#       k>0 covered CORE (200/200/218). Deltas were therefore computed across
#       different item sets and different class proportions. Restricting k=0
#       to the intersection flips the sign of the few-shot effect in several
#       model x task x k cells. Every delta is now paired, and each is
#       accompanied by an exact McNemar test on those same items.
#
#   R3  UNPARSEABLE PREDICTIONS ARE SCORED, NOT DROPPED.
#       `summarise()` computed macro-F1 on `g[g.parse_ok]` only, so a model
#       that emitted an off-taxonomy label had that item deleted from its own
#       denominator - failure was rewarded. Both conventions are now reported:
#       macro_f1_strict (unparseable = wrong, the headline) and
#       macro_f1_parsed_only (the previous behaviour, retained for continuity).
#
#   R4  BOOTSTRAP IS STRATIFIED.
#       The naive bootstrap resampled rows i.i.d. with a fixed `labels=` list,
#       so a class absent from a resample scored F1=0 and was still averaged
#       in. On the 60-item Gemini all-11 cell an average of 1.37 classes (max
#       5) vanished per resample, dragging the lower bound down to an artefact.
#       Resampling within class holds the label distribution fixed, which is
#       the quantity macro-F1 is defined over.
#
#   R5  EVALUABILITY GATE.
#       A macro-F1 whose classes rest on one or two test items is not a
#       measurement. Every cell is checked against MIN_CLASS_SUPPORT and
#       marked reportable / not_reportable with the offending classes named,
#       so a figure can grey them out instead of ranking them.
#
#   R6  PARSER AUDIT (positional vs longest-first).
#       `parse_label` scanned labels longest-first, so "not usability but
#       security" would resolve to usability. This cell re-parses every stored
#       raw_output using earliest-position matching and reports how many
#       predictions change. It does NOT silently overwrite them: the diff is
#       written out for inspection and applied only if APPLY_REPARSE = True.
#
#   R7  LIKE-FOR-LIKE INTERSECTION TABLE.
#       All models re-scored on the exact set of items every model completed,
#       with the per-class support of that intersection emitted alongside, so
#       a comparison can be shown to be valid rather than asserted.
#
#   R8  THE BINARY STORE IS RE-PARSED TOO. 18 stored rows are truncated
#       deliberations naming both classes without answering; they are marked
#       unparseable rather than resolved by substring or position luck.
#
# DELIBERATELY NOT FIXED HERE (cannot be, without new inference):
#   - The CORE subset over-represents rare classes because stratified()'s
#     min_per_class=10 floor is applied after largest-remainder allocation and
#     without renormalisation (portability is 4.1% of CORE vs 1.8% of the
#     corpus, and |CORE| = 218 rather than the requested 200). Mitigation:
#     report FULL-scope numbers as the headline for local models and use CORE
#     only for API comparability. This cell emits both and labels them.
#   - 145 Groq rows lost to rate limiting are missing-not-at-random
#     (maintainability lost 60%). This cell quantifies the loss per class; a
#     top-up run is needed to remove it.
#   - Gemini's 180-request daily cap gave ~60 items per task. R5 will mark its
#     all-11 cell not_reportable.
#
# OUTPUTS -> /kaggle/working/
#   predictions_subtype_repaired.parquet (+ .csv)   store + scope + reparse cols
#   s3b_fix_summary.csv                metrics, both scoring conventions, CIs
#   s3b_fix_fewshot_paired.csv         paired deltas + McNemar
#   s3b_fix_likeforlike.csv            all models on the common intersection
#   s3b_fix_perclass.csv               per-class P/R/F1 + support + reliability
#   s3b_fix_evaluability.csv           reportable / not_reportable per cell
#   s3b_fix_reparse_diff.csv           predictions changed by positional parse
#   s3b_fix_missingness.csv            API dropout per class (MNAR evidence)
#   s3b_fix_binary_reparse_diff.csv    binary predictions changed by R8
#   predictions_llm_repaired.parquet (+ .csv)   binary store, repaired parse
#   s3b_fix_report.md                  human-readable changelog of every delta
# =============================================================================

import glob
import json
import os
import re
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats as sps
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------- configuration
SEED = 42
N_BOOT = 2000            # doubled: stratified resampling has lower variance per draw
MIN_CLASS_SUPPORT = 5    # below this a class cannot carry 1/K of a macro average
# ADOPTED, and now on a measured basis rather than an asserted one. The comment
# used to read "changes 0 of 16,786 sub-type predictions and 0 of 35,049 binary
# ones". The sub-type half is correct and re-verified. The binary half was
# never checked and is false: 18 stored binary predictions turn out to be
# truncated deliberations that name BOTH classes without answering ("We need
# to decide if the requirement is functional or non-functional" - 17 rows on
# openrouter-nemotron, 1 on qwen2.5-7b). The original longest-first parse had
# manufactured NFR out of them; an earlier draft of R8 manufactured FR by
# position instead. Neither is an answer, so R8 now marks them UNPARSEABLE
# (strict scoring counts them wrong either way). Both diffs are written out -
# s3b_fix_reparse_diff.csv and s3b_fix_binary_reparse_diff.csv - so every
# change is inspectable.
APPLY_REPARSE = True     # see R6
OUT = "/kaggle/working"

# Cleland-Huang et al. (2007) PROMISE NFR sub-classes, aligned with ISO/IEC 25010:2011.
CATEGORIES_ALL = ["availability", "fault_tolerance", "legal", "look_and_feel",
                  "maintainability", "operational", "performance", "portability",
                  "scalability", "security", "usability"]
TOP4 = ["security", "usability", "operational", "performance"]
TOP6 = TOP4 + ["look_and_feel", "availability"]
LABELSETS = {"subtype_all": CATEGORIES_ALL, "subtype_top6": TOP6, "subtype_top4": TOP4}

UNPARSED = "__unparsed__"   # sentinel prediction for strict scoring; never a gold label

os.makedirs(OUT, exist_ok=True)
rng_global = np.random.default_rng(SEED)


# ------------------------------------------------------------------------ io
def find_store():
    """Locate the committed Stage 3b prediction store."""
    pats = [
        f"{OUT}/predictions_subtype.parquet",
        "/kaggle/input/**/predictions_subtype.parquet",
        f"{OUT}/predictions_subtype.csv",
        "/kaggle/input/**/predictions_subtype.csv",
        "./predictions_subtype.parquet",
        "./predictions_subtype.csv",
    ]
    for p in pats:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(
        "predictions_subtype.(parquet|csv) not found. Add the committed "
        "stage3b-subtype-harness notebook as an input dataset."
    )


def load_store(path):
    """`keep_default_na=False` is mandatory: the store uses the string sentinel
    'not_applicable' in class_weighting, and pandas would otherwise coerce
    NA-looking tokens and silently drop rows from filtered views."""
    if path.endswith(".parquet"):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, keep_default_na=False, low_memory=False)
    for c in ("shot_k",):
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    df["parse_ok"] = df["parse_ok"].astype(str).str.lower().isin(["true", "1"])
    df["y_true"] = df["y_true"].astype(str)
    df["y_pred"] = df["y_pred"].astype(str)
    return df


STORE_PATH = find_store()
store = load_store(STORE_PATH)
print(f"[load] {STORE_PATH}  ->  {len(store):,} rows, {store.model_tag.nunique()} models")


# ============================================================================
# R1 - reconstruct `scope`
# ============================================================================
def reconstruct_scope(df):
    """CORE is the id-set the few-shot conditions ran on. It is a single fixed
    set per (task, dataset), so it can be recovered exactly from the k=1 rows.
    Both invariants are asserted rather than trusted."""
    core = {}
    for (task, ds), g in df[df.shot_k == 1].groupby(["task", "dataset"]):
        per_model = {m: frozenset(x.id) for m, x in g.groupby("model_tag")}
        assert len(set(per_model.values())) == 1, (
            f"CORE id-set differs across models for {task}/{ds}: "
            f"{ {m: len(v) for m, v in per_model.items()} }. "
            "scope cannot be reconstructed; re-run the harness with scope stored."
        )
        core[(task, ds)] = set(next(iter(per_model.values())))

    df = df.copy()
    # `in_core` is membership, not a partition. The harness deduplicated the
    # CORE and FULL k=0 passes through done_keys, so each item is stored once
    # and CORE is NESTED inside FULL. Labelling the leftover items "full" would
    # make the "full" view the COMPLEMENT of core - a set that never existed as
    # an evaluation and that is missing whole classes (portability = 0 items).
    # Membership plus two explicit views keeps core nested where it belongs.
    df["in_core"] = [
        r.id in core.get((r.task, r.dataset), set())
        for r in df.itertuples(index=False)
    ]

    # every API model must lie inside CORE; if not, the two tiers were never
    # evaluated on comparable items and no like-for-like table is possible.
    for (m, task, ds), g in df[df.model_type != "open_local"].groupby(
            ["model_tag", "task", "dataset"]):
        outside = set(g.id) - core.get((task, ds), set())
        assert not outside, (
            f"{m} on {task}/{ds} has {len(outside)} ids outside CORE."
        )
    return df, core


store, CORE_IDS = reconstruct_scope(store)
print("[R1] scope reconstructed:",
      {t: (g[g.in_core].id.nunique(), g.id.nunique())
       for t, g in store.groupby("task")},
      "(core, full) unique ids per task")


# ============================================================================
# R6 - parser audit: earliest-position matching instead of longest-first
# ============================================================================
_THINK = re.compile(r"<think>.*?</think>", re.S)


def clean_output(raw):
    s = str(raw)
    if "<think>" in s and "</think>" not in s:
        return ""
    return _THINK.sub(" ", s).replace("<think>", " ").replace("</think>", " ").strip()


def parse_label_positional(task, raw):
    """Return the label whose surface form appears EARLIEST in the output.

    The harness scanned longest-label-first, which is position-blind: for
    "security, not maintainability" it returns MAINTAINABILITY, because the
    15-character label is tested before the 8-character one, even though the
    model led with security. Earliest-position returns the label the model
    actually led with. (The example this docstring used to give - "not
    usability but security" - was a bad one: usability is both the longest
    match AND the earliest, so both rules return it and the example
    demonstrated nothing.) Ties at the same start offset are broken by the
    longer surface form, which preserves the original protection against
    'fault_tolerance' being shadowed by a shorter substring."""
    if raw is None:
        return None, False
    r = clean_output(raw).lower()
    if not r:
        return None, False
    best = None
    for c in LABELSETS[task]:
        for surface in (c, c.replace("_", " ")):
            i = r.find(surface)
            if i >= 0 and (best is None or i < best[0] or
                           (i == best[0] and len(surface) > best[1])):
                best = (i, len(surface), c)
    return (best[2], True) if best else (None, False)


rep = store.apply(
    lambda r: parse_label_positional(r.task, r.raw_output), axis=1, result_type="expand")
store["y_pred_v2"] = rep[0].fillna(UNPARSED).astype(str)
store["parse_ok_v2"] = rep[1].astype(bool)

diff = store[(store.parse_ok | store.parse_ok_v2) &
             (store.y_pred.where(store.parse_ok, UNPARSED) != store.y_pred_v2)]
diff[["model_tag", "task", "shot_k", "prompt_id", "id", "y_true",
      "y_pred", "y_pred_v2", "raw_output"]].to_csv(
    f"{OUT}/s3b_fix_reparse_diff.csv", index=False)
print(f"[R6] positional re-parse changes {len(diff)} / {len(store)} predictions "
      f"({100*len(diff)/len(store):.2f}%) -> s3b_fix_reparse_diff.csv")

if APPLY_REPARSE:
    store["y_pred"] = store["y_pred_v2"]
    store["parse_ok"] = store["parse_ok_v2"]
    print("[R6] APPLY_REPARSE=True: positional parse adopted as canonical.")

# strict prediction column: unparseable rows keep a sentinel instead of vanishing
store["y_pred_strict"] = store["y_pred"].where(store["parse_ok"], UNPARSED)


# ============================================================================
# R4 - stratified bootstrap
# ============================================================================
def _macro_f1_fast(t_idx, p_idx, K, n_boot_rows=None):
    """Vectorised macro-F1 from integer-encoded labels.

    sklearn's f1_score inside a 2000-iteration loop over ~100 cells costs
    minutes; counting true-positives / predicted / actual with bincount is
    numerically identical and runs in milliseconds. `t_idx`/`p_idx` may be 2-D
    (n_boot, n) to score every bootstrap replicate in one pass. Predictions
    outside the label set (the UNPARSED sentinel) are encoded as K and are
    counted as a miss for the true class without creating a class of their own,
    which is exactly the strict convention."""
    if t_idx.ndim == 1:
        t_idx, p_idx = t_idx[None, :], p_idx[None, :]
    B, n = t_idx.shape
    off = np.arange(B)[:, None] * (K + 1)
    tp = np.bincount((t_idx + off)[t_idx == p_idx].ravel(),
                     minlength=B * (K + 1)).reshape(B, K + 1)[:, :K]
    pred = np.bincount((p_idx + off).ravel(), minlength=B * (K + 1)).reshape(B, K + 1)[:, :K]
    act = np.bincount((t_idx + off).ravel(), minlength=B * (K + 1)).reshape(B, K + 1)[:, :K]
    denom = pred + act
    f1 = np.divide(2.0 * tp, denom, out=np.zeros_like(denom, dtype=float),
                   where=denom > 0)
    return f1.mean(axis=1)


def stratified_bootstrap_ci(y_true, y_pred, labelset, n_boot=N_BOOT, seed=SEED):
    """Resample WITHIN each true class, holding class sizes fixed.

    macro-F1 averages over a declared label set, so a resample that loses a
    class scores it 0 and still divides by K - an artefact of the resampling
    scheme, not of the model. Conditioning on the label distribution removes
    it. Classes with a single item are degenerate under any resampling scheme;
    R5 refuses to report those cells rather than pretending the interval is
    meaningful."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if len(y_true) < 10:
        return np.nan, np.nan
    K = len(labelset)
    code = {c: i for i, c in enumerate(labelset)}
    t = np.array([code.get(v, K) for v in y_true])
    p = np.array([code.get(v, K) for v in y_pred])
    rng = np.random.default_rng(seed)
    idx_by_cls = [np.flatnonzero(t == i) for i in range(K)]
    idx_by_cls = [ix for ix in idx_by_cls if len(ix)]
    take = np.concatenate(
        [rng.choice(ix, size=(n_boot, len(ix)), replace=True) for ix in idx_by_cls],
        axis=1)
    stat = _macro_f1_fast(t[take], p[take], K)
    lo, hi = np.percentile(stat, [2.5, 97.5])
    return round(float(lo), 4), round(float(hi), 4)


# ============================================================================
# R5 - evaluability gate
# ============================================================================
def evaluability(y_true, labelset, min_support=MIN_CLASS_SUPPORT):
    """A cell is reportable only if every label in its declared label set has
    at least `min_support` test items. Returns (ok, min_support_seen, offenders)."""
    counts = pd.Series(y_true).value_counts()
    supports = {c: int(counts.get(c, 0)) for c in labelset}
    bad = {c: n for c, n in supports.items() if n < min_support}
    return (len(bad) == 0), min(supports.values()), bad


# ============================================================================
# metrics per cell (both scoring conventions)
# ============================================================================
def cell_metrics(g, labelset):
    ok = g[g.parse_ok]
    yt_s, yp_s = g.y_true.to_numpy(), g.y_pred_strict.to_numpy()
    yt_p, yp_p = ok.y_true.to_numpy(), ok.y_pred.to_numpy()

    f_strict = f1_score(yt_s, yp_s, labels=labelset, average="macro", zero_division=0)
    f_parsed = (f1_score(yt_p, yp_p, labels=labelset, average="macro", zero_division=0)
                if len(ok) else np.nan)
    lo, hi = stratified_bootstrap_ci(yt_s, yp_s, labelset)
    rep_ok, min_sup, offenders = evaluability(yt_s, labelset)

    return {
        "n": len(g),
        "n_parsed": int(g.parse_ok.sum()),
        "parse_rate": round(float(g.parse_ok.mean()), 4),
        "macro_f1_strict": round(float(f_strict), 4),
        "macro_f1_parsed_only": round(float(f_parsed), 4) if len(ok) else np.nan,
        "delta_strict_minus_parsed": (round(float(f_strict - f_parsed), 4)
                                      if len(ok) else np.nan),
        "ci_lo_strat": lo, "ci_hi_strat": hi,
        "ci_width": round(hi - lo, 4) if not np.isnan(hi) else np.nan,
        "weighted_f1_strict": round(float(f1_score(
            yt_s, yp_s, labels=labelset, average="weighted", zero_division=0)), 4),
        "accuracy_strict": round(float(accuracy_score(yt_s, yp_s)), 4),
        "min_class_support": min_sup,
        "reportable": bool(rep_ok),
        "underpowered_classes": ";".join(f"{k}={v}" for k, v in sorted(offenders.items())),
    }


KEYS = ["model_tag", "model_type", "task", "dataset", "split",
        "prompt_id", "shot_k"]
rows = []
for kv, g in store.groupby(KEYS, dropna=False):
    labelset = LABELSETS[dict(zip(KEYS, kv))["task"]]
    # Two NESTED views of the same cell:
    #   full  - every item this model answered (the headline for local models,
    #           and the only view whose class proportions match the corpus)
    #   core  - the fixed shared subset, the only view comparable across tiers
    # A cell already wholly inside CORE (every API model, every few-shot
    # condition) would otherwise be emitted twice under two names.
    # A cell wholly inside CORE never saw the full frame, so the single view
    # it gets is labelled "core" (the old "full" label on those rows
    # contradicted the definition above); covers_full_frame separates a true
    # full-frame cell from a quota-capped one either way.
    views = ([("core", g)] if g.in_core.all()
             else [("full", g), ("core", g[g.in_core])])
    for view, sub in views:
        if len(sub) == 0:
            continue
        rec = dict(zip(KEYS, kv))
        rec["view"] = view
        rec["covers_full_frame"] = bool(not g.in_core.all())
        rec.update(cell_metrics(sub, labelset))
        if "cost_usd" in sub.columns:
            rec["cost_usd"] = round(
                float(pd.to_numeric(sub.cost_usd, errors="coerce").sum()), 6)
            rec["latency_s_mean"] = round(
                float(pd.to_numeric(sub.latency_s, errors="coerce").mean()), 4)
        rows.append(rec)

summary = pd.DataFrame(rows).sort_values(
    ["task", "view", "split", "shot_k", "macro_f1_strict"],
    ascending=[True, True, True, True, False])
summary.to_csv(f"{OUT}/s3b_fix_summary.csv", index=False)
print(f"[R3/R4/R5] summary: {len(summary)} cells -> s3b_fix_summary.csv")

evalu = summary[["model_tag", "task", "view", "split", "shot_k", "n",
                 "macro_f1_strict", "min_class_support", "reportable",
                 "underpowered_classes"]]
evalu.to_csv(f"{OUT}/s3b_fix_evaluability.csv", index=False)
n_bad = int((~summary.reportable).sum())
print(f"[R5] {n_bad} / {len(summary)} cells fall below "
      f"MIN_CLASS_SUPPORT={MIN_CLASS_SUPPORT} and are marked not reportable.")


# ============================================================================
# R2 - paired few-shot with exact McNemar
# ============================================================================
def mcnemar_exact(correct_a, correct_b):
    """Exact binomial McNemar. b+c is small in several cells here, where the
    chi-square approximation is unreliable, so the exact test is used
    throughout for consistency rather than switching test by sample size."""
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    if b + c == 0:
        return b, c, 1.0
    p = float(sps.binomtest(min(b, c), b + c, 0.5).pvalue)
    return b, c, p


fs_rows = []
base = store[store.prompt_id == "base"]
for (m, task, ds), g in base.groupby(["model_tag", "task", "dataset"]):
    ks = {int(k): x for k, x in g.groupby("shot_k")}
    if 0 not in ks or len(ks) < 2:
        continue
    labelset = LABELSETS[task]
    shot_ks = sorted(k for k in ks if k > 0)
    common = set.intersection(*[set(ks[k].id) for k in [0] + shot_ks])
    if len(common) < 30:
        continue

    z_full = ks[0]
    z = ks[0][ks[0].id.isin(common)].sort_values("id")
    f0_full = f1_score(z_full.y_true, z_full.y_pred_strict, labels=labelset,
                       average="macro", zero_division=0)
    f0_pair = f1_score(z.y_true, z.y_pred_strict, labels=labelset,
                       average="macro", zero_division=0)

    for k in shot_ks:
        s = ks[k][ks[k].id.isin(common)].sort_values("id")
        assert (s.id.values == z.id.values).all(), "pairing misaligned"
        fk = f1_score(s.y_true, s.y_pred_strict, labels=labelset,
                      average="macro", zero_division=0)
        b, c, p = mcnemar_exact(
            (z.y_pred_strict.values == z.y_true.values),
            (s.y_pred_strict.values == s.y_true.values))
        fs_rows.append({
            "model_tag": m, "task": task, "dataset": ds, "k": k,
            "n_paired": len(common),
            "n_k0_unpaired": len(z_full),
            # NOT "as_published". This is the FULL-frame k=0 recomputed under
            # the strict convention (unparseable = wrong). What the harness
            # actually published was parsed-only - the convention R3 above
            # replaces - so a column called "as_published" held a number that
            # was never published, and the report table headed it "gain
            # published". Named for what it is.
            "k0_f1_full_frame_strict": round(float(f0_full), 4),
            "k0_f1_paired": round(float(f0_pair), 4),
            "k_f1": round(float(fk), 4),
            "gain_full_frame_strict": round(float(fk - f0_full), 4),
            "gain_paired": round(float(fk - f0_pair), 4),
            "sign_flipped": bool(np.sign(fk - f0_full) != np.sign(fk - f0_pair)),
            "zeroshot_only_correct": b, "fewshot_only_correct": c,
            "mcnemar_p": round(p, 5), "sig_0.05": bool(p < 0.05),
        })

fewshot = pd.DataFrame(fs_rows)
if len(fewshot):
    # Holm correction within the few-shot family of tests
    p = fewshot.mcnemar_p.values
    order = np.argsort(p)
    adj = np.minimum(np.maximum.accumulate(
        p[order] * (len(p) - np.arange(len(p)))), 1.0)
    holm = np.empty(len(p)); holm[order] = adj
    fewshot["mcnemar_p_holm"] = holm.round(5)
    fewshot["sig_holm"] = fewshot.mcnemar_p_holm < 0.05
fewshot.to_csv(f"{OUT}/s3b_fix_fewshot_paired.csv", index=False)
print(f"[R2] few-shot: {len(fewshot)} paired comparisons, "
      f"{int(fewshot.sign_flipped.sum()) if len(fewshot) else 0} sign flips vs "
      f"the unpaired full-frame k=0 baseline -> s3b_fix_fewshot_paired.csv")


# ============================================================================
# R7 - like-for-like intersection across all models
# ============================================================================
lf_rows = []
zs = store[(store.prompt_id == "base") & (store.shot_k == 0)]
for (task, ds), g in zs.groupby(["task", "dataset"]):
    labelset = LABELSETS[task]
    sets = {m: set(x.id) for m, x in g.groupby("model_tag")}
    common = set.intersection(*sets.values())
    sup = g[g.id.isin(common)].drop_duplicates("id").y_true.value_counts()
    rep_ok, min_sup, offenders = evaluability(
        g[g.id.isin(common)].drop_duplicates("id").y_true, labelset)
    for m, x in g.groupby("model_tag"):
        own = f1_score(x.y_true, x.y_pred_strict, labels=labelset,
                       average="macro", zero_division=0)
        cc = x[x.id.isin(common)]
        cf = f1_score(cc.y_true, cc.y_pred_strict, labels=labelset,
                      average="macro", zero_division=0)
        lo, hi = stratified_bootstrap_ci(cc.y_true.to_numpy(),
                                         cc.y_pred_strict.to_numpy(), labelset)
        lf_rows.append({
            "task": task, "dataset": ds, "model_tag": m,
            "n_own": len(x), "f1_own_sample": round(float(own), 4),
            "n_common": len(cc), "f1_common_sample": round(float(cf), 4),
            "delta": round(float(cf - own), 4),
            "ci_lo": lo, "ci_hi": hi,
            "intersection_reportable": bool(rep_ok),
            "intersection_min_support": min_sup,
            "intersection_support": ";".join(
                f"{k}={int(v)}" for k, v in sup.sort_index().items()) or "none",
            "intersection_missing_classes": ";".join(
                sorted(set(labelset) - set(sup.index))) or "none",
        })
likeforlike = pd.DataFrame(lf_rows).sort_values(
    ["task", "f1_common_sample"], ascending=[True, False])
likeforlike.to_csv(f"{OUT}/s3b_fix_likeforlike.csv", index=False)
print("[R7] like-for-like table -> s3b_fix_likeforlike.csv")


# ============================================================================
# per-class table with reliability flags
# ============================================================================
pc_rows = []
pc_iter = []
for (m, task, split, k), g in store[store.prompt_id == "base"].groupby(
        ["model_tag", "task", "split", "shot_k"]):
    # Same rule the summary uses: a cell wholly inside CORE gets ONE row,
    # labelled "core". Labelling it "full" here while the summary labelled the
    # identical cell "core" put two contradictory descriptions of one
    # measurement in two neighbouring files.
    if g.in_core.all():
        pc_iter.append((m, task, "core", split, k, g))
    else:
        pc_iter.append((m, task, "full", split, k, g))
        pc_iter.append((m, task, "core", split, k, g[g.in_core]))
for (m, task, view, split, k, g) in pc_iter:
    labelset = LABELSETS[task]
    pr, rc, f1v, sup = precision_recall_fscore_support(
        g.y_true, g.y_pred_strict, labels=labelset, zero_division=0)
    for c, p_, r_, f_, s_ in zip(labelset, pr, rc, f1v, sup):
        pc_rows.append({
            "model_tag": m, "task": task, "view": view, "split": split,
            "shot_k": int(k), "category": c,
            "precision": round(float(p_), 4), "recall": round(float(r_), 4),
            "f1": round(float(f_), 4), "support": int(s_),
            "reliable": bool(s_ >= MIN_CLASS_SUPPORT),
        })
perclass = pd.DataFrame(pc_rows)
perclass.to_csv(f"{OUT}/s3b_fix_perclass.csv", index=False)
print("[perclass] -> s3b_fix_perclass.csv")


# ============================================================================
# missingness (MNAR evidence for the API tier)
# ============================================================================
miss_rows = []
fail_path = None
for p in [f"{OUT}/stage3b_failures.csv", "/kaggle/input/**/stage3b_failures.csv",
          "./stage3b_failures.csv", os.path.join(os.path.dirname(STORE_PATH),
                                                 "stage3b_failures.csv")]:
    hits = sorted(glob.glob(p, recursive=True))
    if hits:
        fail_path = hits[0]
        break
if fail_path:
    fails = pd.read_csv(fail_path, keep_default_na=False)
    gold = store.drop_duplicates("id").set_index("id").y_true
    fails["y_true"] = fails["id"].map(gold)
    for (m, task), fg in fails.groupby(["model_tag", "task"]):
        scored = store[(store.model_tag == m) & (store.task == task) &
                       (store.shot_k == 0) & (store.prompt_id == "base")]
        sc = scored.y_true.value_counts()
        fc = fg.y_true.value_counts()
        for c in LABELSETS[task]:
            att = int(sc.get(c, 0)) + int(fc.get(c, 0))
            if att == 0:
                continue
            miss_rows.append({
                "model_tag": m, "task": task, "category": c,
                "attempted": att, "scored": int(sc.get(c, 0)),
                "dropped": int(fc.get(c, 0)),
                "dropout_pct": round(100 * int(fc.get(c, 0)) / att, 1),
            })
missing = pd.DataFrame(miss_rows)
missing.to_csv(f"{OUT}/s3b_fix_missingness.csv", index=False)
if len(missing):
    worst = missing.sort_values("dropout_pct", ascending=False).head(3)
    print("[MNAR] worst per-class API dropout:")
    print(worst[["model_tag", "task", "category", "dropped",
                 "attempted", "dropout_pct"]].to_string(index=False))


# ============================================================================
# repaired store + human-readable report
# ============================================================================
store.to_csv(f"{OUT}/predictions_subtype_repaired.csv", index=False)
try:
    store.to_parquet(f"{OUT}/predictions_subtype_repaired.parquet", index=False)
except Exception as e:                                    # pyarrow absent
    print(f"[warn] parquet write skipped: {e}")

lines = [
    "# Stage 3b - repair report",
    "",
    f"Source store: `{STORE_PATH}` ({len(store):,} rows)",
    f"MIN_CLASS_SUPPORT = {MIN_CLASS_SUPPORT}; bootstrap = stratified, "
    f"{N_BOOT} replicates; APPLY_REPARSE = {APPLY_REPARSE}",
    "",
    "## R1 scope reconstruction",
    "",
    "| task | core ids | full ids |",
    "|---|---|---|",
]
for task in sorted(store.task.unique()):
    t = store[store.task == task]
    lines.append(f"| {task} | {t[t.in_core].id.nunique()} | "
                 f"{t.id.nunique()} |")

lines += ["", "## R2 few-shot sign flips introduced by unpaired k=0", ""]
if len(fewshot) and fewshot.sign_flipped.any():
    lines += ["| model | task | k | gain (full frame, strict) | gain (paired) |",
              "|---|---|---|---|---|"]
    for r in fewshot[fewshot.sign_flipped].itertuples():
        lines.append(f"| {r.model_tag} | {r.task} | {r.k} | "
                     f"{r.gain_full_frame_strict:+.4f} | {r.gain_paired:+.4f} |")
else:
    lines.append("None.")

lines += ["", "## R3 effect of strict scoring (unparseable = wrong)", ""]
aff = summary[(summary.parse_rate < 1.0) & summary.delta_strict_minus_parsed.notna()]
if len(aff):
    lines += ["| model | task | view | prompt | k | parse rate | parsed-only | strict | delta |",
              "|---|---|---|---|---|---|---|---|---|"]
    for r in aff.sort_values("delta_strict_minus_parsed").itertuples():
        lines.append(f"| {r.model_tag} | {r.task} | {r.view} | {r.prompt_id} | {r.shot_k} | "
                     f"{r.parse_rate:.3f} | {r.macro_f1_parsed_only:.4f} | "
                     f"{r.macro_f1_strict:.4f} | {r.delta_strict_minus_parsed:+.4f} |")
else:
    lines.append("No cell had parse failures.")

lines += ["", f"## R5 cells not reportable at min support {MIN_CLASS_SUPPORT}", ""]
nr = summary[~summary.reportable]
if len(nr):
    lines += ["| model | task | view | n | macro-F1 (strict) | min support | underpowered |",
              "|---|---|---|---|---|---|---|"]
    for r in nr.sort_values(["task", "model_tag"]).itertuples():
        lines.append(f"| {r.model_tag} | {r.task} | {r.view} | {r.n} | "
                     f"{r.macro_f1_strict:.4f} | {r.min_class_support} | "
                     f"{r.underpowered_classes} |")
else:
    lines.append("All cells reportable.")

lines += ["", "## R6 parser audit", "",
          f"Positional matching changes {len(diff)} of {len(store)} predictions "
          f"({100*len(diff)/len(store):.2f}%). Applied: {APPLY_REPARSE}."]

with open(f"{OUT}/s3b_fix_report.md", "w") as fh:
    fh.write("\n".join(lines) + "\n")

# ============================================================================
# R8 - THE BINARY STORE, brought onto the same convention as the sub-type one
#
# R6 above re-parses the sub-type store; the binary store still carried the
# harness's longest-first parse. Auditing it surfaced 18 rows whose raw
# output is a truncated deliberation naming BOTH classes without deciding
# ("We need to decide if the requirement is functional or non-functional").
# The longest-first parse had turned those into NFR; earliest-position
# matching would turn them into FR. Both manufacture a label from a
# non-answer, so _parse_binary refuses both-sided outputs outright and the 17
# rows become UNPARSEABLE - which strict scoring already counts as wrong.
# The repaired binary store gets the same y_pred_strict column the sub-type
# store gets; Stage 4 consumes it unchanged. No API call, no GPU.
#
# Scale, so the change is not mistaken for something larger: 17 of the 18
# rows belong to openrouter-nemotron, whose cells the evaluability gate
# already excludes (n=21 per cell), and 1 to qwen2.5-7b (one item of a
# 960-item frame). Headline numbers move by at most one item's worth; the
# point is that the store no longer contains labels no model produced.
# ============================================================================
BINARY_LABELSETS = {"fr_nfr": ["FR", "NFR"],
                    "security": ["security", "non-security"]}


def _parse_binary(task, raw):
    """Binary parse: leading-label first, then single-side, else refuse.

    Position alone cannot rescue a binary answer that names both classes:
    every row the first R8 audit changed turned out to be a truncated
    deliberation ("We need to decide if the requirement is functional or
    non-functional") that never answered. Earliest-position matching turned
    those non-answers into FR, just as longest-first had turned them into
    NFR. So: a label that OPENS the answer is the decision (explanations
    after it may name the other class); otherwise the answer must name
    exactly one class, and both-sided outputs are unparseable. Short forms
    match on word boundaries ("fr" never fires inside "free")."""
    if raw is None:
        return None, False
    r = clean_output(raw).lower()
    if not r:
        return None, False

    # Word-boundary matcher: "fr" must not fire inside "free" or "from", and
    # "security" must not fire as the tail of "non-security".
    def _word(sub):
        return [m.start() for m in
                re.finditer(r"(?<![a-z0-9])" + re.escape(sub) + r"(?![a-z0-9])", r)]

    if task == "security":
        neg = _word("non-security") + _word("non security") + _word("nonsecurity")
        pos = [i for i in _word("security")
               if r[max(0, i - 4):i] not in ("non-", "non ") and r[max(0, i - 3):i] != "non"]
        sides = [("non-security", neg), ("security", pos)]
    else:
        nfr = (_word("nfr") + _word("non-functional") + _word("nonfunctional")
               + _word("non functional"))
        fr = _word("fr") + [i for i in _word("functional")
                            if r[max(0, i - 4):i] not in ("non-", "non ")
                            and r[max(0, i - 3):i] != "non"]
        sides = [("NFR", nfr), ("FR", fr)]

    # 1. A LEADING label is the decision, whatever the explanation after it
    #    mentions: "non-security\n\nThis is about functionality rather than
    #    security" answers non-security.
    leading = [(lab, hits) for lab, hits in sides if hits and min(hits) == 0]
    if len(leading) == 1:
        return leading[0][0], True
    # 2. No leading label: exactly one side mentioned anywhere -> that side.
    #    BOTH sides mentioned is an enumeration, not a decision ("We need to
    #    decide if the requirement is functional or non-functional") - the
    #    old longest-first scan silently turned 18 such stored rows into
    #    labels; they are refused instead.
    named = [(lab, hits) for lab, hits in sides if hits]
    if len(named) == 1:
        return named[0][0], True
    return None, False


_bin_hits = []
for _p in [f"{OUT}/predictions_llm.parquet", "/kaggle/input/**/predictions_llm.parquet",
           f"{OUT}/predictions_llm.csv", "/kaggle/input/**/predictions_llm.csv"]:
    _bin_hits = sorted(glob.glob(_p, recursive=True))
    if _bin_hits:
        break
if not _bin_hits:
    print("[R8] predictions_llm.* not attached - binary store not repaired. "
          "Stage 4 will read the harness store as-is, which still carries the "
          "longest-first parse. Attach stage3-llm-harness to close this.")
else:
    bstore = load_store(_bin_hits[0])
    _rep = bstore.apply(lambda r: _parse_binary(r.task, r.raw_output),
                        axis=1, result_type="expand")
    bstore["y_pred_v2"] = _rep[0].fillna(UNPARSED).astype(str)
    bstore["parse_ok_v2"] = _rep[1].astype(bool)
    _bdiff = bstore[(bstore.parse_ok | bstore.parse_ok_v2) &
                    (bstore.y_pred.where(bstore.parse_ok, UNPARSED) != bstore.y_pred_v2)]
    _bdiff[["model_tag", "task", "shot_k", "prompt_id", "id", "y_true",
            "y_pred", "y_pred_v2", "raw_output"]].to_csv(
        f"{OUT}/s3b_fix_binary_reparse_diff.csv", index=False)
    print(f"[R8] binary store: positional re-parse changes {len(_bdiff)} / "
          f"{len(bstore)} predictions ({100*len(_bdiff)/len(bstore):.2f}%)"
          f" -> s3b_fix_binary_reparse_diff.csv")
    if len(_bdiff):
        print("     by model:", _bdiff.model_tag.value_counts().to_dict())
    if APPLY_REPARSE:
        bstore["y_pred"] = bstore["y_pred_v2"]
        bstore["parse_ok"] = bstore["parse_ok_v2"]
    bstore["y_pred_strict"] = bstore["y_pred"].where(bstore["parse_ok"], UNPARSED)
    bstore.to_parquet(f"{OUT}/predictions_llm_repaired.parquet", index=False)
    bstore.to_csv(f"{OUT}/predictions_llm_repaired.csv", index=False)
    print(f"[R8] wrote predictions_llm_repaired.parquet ({len(bstore):,} rows). "
          f"Stage 4 and Stage 5 prefer this over the harness store.")

print("\n[done] wrote:")
for f in sorted(glob.glob(f"{OUT}/s3b_fix_*")
                + glob.glob(f"{OUT}/predictions_subtype_repaired.*")
                + glob.glob(f"{OUT}/predictions_llm_repaired.*")):
    print("   ", os.path.basename(f), f"{os.path.getsize(f)/1024:.1f} KB")


ATTACHED INPUTS

unified.parquet   <- Stage 1 corpus        (Stages 2, 4)
   (not attached)

splits.json   <- Stage 1 splits        (Stage 2)
   (not attached)

predictions_finetuned.parquet   <- Stage 2 encoder store (Stage 2 resume, 4, 5)
   (not attached)

predictions_llm.parquet   <- Stage 3 LLM binary    (Stages 4, 5)
   /kaggle/input/datasets/zahrasamir/stage3-llm-harness/predictions_llm.parquet
           823,810 bytes   2026-08-27 08:44  >> THIS ONE WILL BE USED

predictions_subtype.parquet   <- Stage 3b raw subtype  (Stage 3b-repair)
   /kaggle/input/datasets/zahrasamir/stage3b-subtype-outputs/predictions_subtype.parquet
           411,559 bytes   2026-08-27 08:44  >> THIS ONE WILL BE USED

predictions_subtype_repaired.parquet   <- Stage 3b repaired    (Stages 4, 5)
   (not attached)

tab9_evaluability.csv   <- Stage 4 gate          (Stage 5)
   (not attached)

ROW COUNTS OF WHAT WILL ACTUALLY BE READ
  predictions_llm.parquet                 35,049 rows   expected 35,049   OK